# SaaS/E-Commerce Cohort Retention & Customer Lifetime Value (CLTV) Analysis

## Notebook 1: Data Cleaning & Preparation

### Project Overview

Customer retention is one of the strongest indicators of sustainable business growth. While acquiring new customers drives expansion, retaining existing customers improves profitability and long-term customer value.

This notebook performs the data acquisition, exploration, and preprocessing required for Cohort Analysis and Customer Lifetime Value (CLTV) analysis. The cleaned dataset produced here will be used throughout the remaining stages of the project.

---

### Objectives

- Load and inspect the raw transactional dataset.
- Assess data quality.
- Handle missing and invalid records.
- Remove cancelled or refunded transactions where appropriate.
- Engineer features required for cohort analysis.
- Save a clean dataset for downstream analysis.

---

### Dataset

**Dataset:** Online Retail II

**Source:** UCI Machine Learning Repository

**Period Covered:** December 2009 – December 2011

**Records:** 1,067,371

**Features:** 8

In [1]:
# ==========================================
# Import Required Libraries
# ==========================================

import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

# Plot Settings
plt.style.use("default")
sns.set_theme(style="whitegrid")

# Ignore unnecessary warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ==========================================
# Project Configuration
# ==========================================

from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

# Data folders
RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

print("Project Root:", PROJECT_ROOT)
print("Raw Data:", RAW_DATA)
print("Processed Data:", PROCESSED_DATA)

Project Root: c:\Users\HP\OneDrive\Documents\My Git Project\SaaS-ECommerce-Cohort-Retention-CLTV-Analysis
Raw Data: c:\Users\HP\OneDrive\Documents\My Git Project\SaaS-ECommerce-Cohort-Retention-CLTV-Analysis\data\raw
Processed Data: c:\Users\HP\OneDrive\Documents\My Git Project\SaaS-ECommerce-Cohort-Retention-CLTV-Analysis\data\processed


In [3]:
df = pd.read_csv(
    RAW_DATA / "online_retail_II.csv",
    encoding="ISO-8859-1"
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [4]:
# ==========================================
# Dataset Overview
# ==========================================

print("=" * 50)
print("Dataset Shape")
print("=" * 50)
print(df.shape)

print("\nColumns")
print(df.columns.tolist())

print("\nData Types")
display(df.dtypes)

print("\nFirst Five Records")
display(df.head())

Dataset Shape
(1067371, 8)

Columns
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Data Types


Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object


First Five Records


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


In [5]:
# ==========================================
# Dataset Dimensions
# ==========================================

print("=" * 60)
print("DATASET DIMENSIONS")
print("=" * 60)

rows, cols = df.shape

print(f"Rows    : {rows:,}")
print(f"Columns : {cols}")

DATASET DIMENSIONS
Rows    : 1,067,371
Columns : 8


In [6]:
# ==========================================
# Column Names
# ==========================================

print("=" * 60)
print("COLUMN NAMES")
print("=" * 60)

for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

COLUMN NAMES
1. Invoice
2. StockCode
3. Description
4. Quantity
5. InvoiceDate
6. Price
7. Customer ID
8. Country


In [7]:
# ==========================================
# First Five Records
# ==========================================

display(df.head())

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


In [8]:
# ==========================================
# Random Sample
# ==========================================

display(df.sample(10, random_state=42))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
455941,532657,21314,SMALL GLASS HEART TRINKET POT,12,2010-11-14 11:10:00,2.10,"14,562.00",United Kingdom
826291,563214,22383,LUNCH BAG SUKI DESIGN,2,2011-08-14 12:56:00,1.65,"16,370.00",United Kingdom
191636,507597,22561,WOODEN SCHOOL COLOURING SET,12,2010-05-10 13:21:00,1.65,"17,700.00",United Kingdom
25864,491634,21588,RETRO SPOT GIANT TUBE MATCHES,1,2009-12-11 15:40:00,2.55,"17,841.00",United Kingdom
73233,496007,85232B,SET/3 RUSSIAN DOLL STACKING TINS,3,2010-01-28 12:32:00,4.95,"15,203.00",United Kingdom
557542,539041,21832,CHOCOLATE CALCULATOR,4,2010-12-15 15:34:00,1.65,"15,456.00",United Kingdom
985772,575905,22089,PAPER BUNTING VINTAGE PAISLEY,6,2011-11-11 15:49:00,2.95,"13,732.00",United Kingdom
568585,540026,21519,GIN & TONIC DIET GREETING CARD,2,2011-01-04 13:25:00,0.85,NaN,United Kingdom
530265,536804,22988,SOLDIERS EGG CUP,72,2010-12-02 16:34:00,1.25,"14,031.00",United Kingdom
649024,546899,20719,WOODLAND CHARLOTTE BAG,50,2011-03-17 18:27:00,0.72,"14,298.00",United Kingdom


In [9]:
# ==========================================
# Dataset Information
# ==========================================

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


In [10]:
# ==========================================
# Numerical Summary
# ==========================================

display(df.describe())

,Quantity,Price,Customer ID
count,"1,067,371.00","1,067,371.00","824,364.00"
mean,9.94,4.65,"15,324.64"
std,172.71,123.55,"1,697.46"
min,"-80,995.00","-53,594.36","12,346.00"
25%,1.00,1.25,"13,975.00"
50%,3.00,2.10,"15,255.00"
75%,10.00,4.15,"16,797.00"
max,"80,995.00","38,970.00","18,287.00"


In [11]:
# ==========================================
# Categorical Summary
# ==========================================

display(df.describe(include="object"))

,Invoice,StockCode,Description,InvoiceDate,Country
count,1067371,1067371,1062989,1067371,1067371
unique,53628,5305,5698,47635,43
top,537434,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2010-12-06 16:57:00,United Kingdom
freq,1350,5829,5918,1350,981330


In [12]:
# ==========================================
# Missing Values
# ==========================================

missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

missing = missing[missing["Missing Values"] > 0]

display(missing.sort_values("Percentage", ascending=False))

,Missing Values,Percentage
Customer ID,243007,22.77
Description,4382,0.41


In [13]:
# ==========================================
# Duplicate Records
# ==========================================

duplicates = df.duplicated().sum()

print(f"Duplicate Rows: {duplicates:,}")

Duplicate Rows: 34,335


In [14]:
# ==========================================
# Cancelled Transactions
# ==========================================

cancelled = df["Invoice"].str.startswith("C").sum()

print(f"Cancelled Transactions: {cancelled:,}")
print(f"Percentage: {cancelled / len(df) * 100:.2f}%")

Cancelled Transactions: 19,494
Percentage: 1.83%


In [15]:
# ==========================================
# Negative Quantities
# ==========================================

negative_qty = (df["Quantity"] < 0).sum()

print(f"Negative Quantities: {negative_qty:,}")

Negative Quantities: 22,950


In [16]:
# ==========================================
# Negative Prices
# ==========================================

negative_price = (df["Price"] < 0).sum()

print(f"Negative Prices: {negative_price:,}")

Negative Prices: 5


In [17]:
# ==========================================
# Zero Prices
# ==========================================

zero_price = (df["Price"] == 0).sum()

print(f"Zero Price Transactions: {zero_price:,}")

Zero Price Transactions: 6,202


In [18]:
# Missing values
missing

# Duplicate rows
duplicates

# Cancelled invoices
cancelled

# Negative quantities
negative_qty

# Negative prices
negative_price

# Zero prices
zero_price

np.int64(6202)

## Data Profiling Summary

### Overview

An initial data profiling exercise was conducted to understand the structure, completeness, and overall quality of the **Online Retail II** dataset before commencing data cleaning and preprocessing. This step is essential for identifying data quality issues, validating the dataset against project requirements, and informing the data cleaning strategy for subsequent cohort and Customer Lifetime Value (CLTV) analyses.

---

### Dataset Overview

| Metric | Value |
|---------|------:|
| Total Records | 1,067,371 |
| Total Columns | 8 |
| Memory Usage | 65.1 MB |
| Time Period | December 2009 – December 2011 |
| Countries | 43 |
| Unique Invoices | 53,628 |
| Unique Products (StockCode) | 5,305 |
| Unique Product Descriptions | 5,698 |
| Unique Invoice Dates | 47,635 |

The dataset provides a comprehensive transactional history suitable for customer retention, cohort, and lifetime value analyses.

---

### Data Structure Assessment

The dataset contains eight variables describing customer transactions, including invoice information, product details, transaction quantity, unit price, customer identifier, transaction timestamp, and customer location.

Most variables were loaded with the expected data types. However, the **InvoiceDate** column was imported as a string (`object`) instead of a datetime object and will require conversion before any time-based analysis can be performed.

---

### Missing Value Assessment

The profiling identified missing values in two variables:

| Column | Missing Records | Observation |
|---------|----------------:|------------|
| Customer ID | 243,007 | Significant missing values |
| Description | 4,382 | Minimal missing values |

The missing **Customer ID** values represent approximately **22.8%** of the dataset. Since customer identification is fundamental for cohort assignment and CLTV calculations, records without customer IDs cannot be used for customer-level behavioural analysis.

The missing product descriptions represent less than one percent of the dataset and are not expected to materially affect the objectives of this project.

---

### Numerical Data Assessment

The descriptive statistics revealed several characteristics of the transactional data:

- Transaction quantities range from **-80,995** to **80,995** units.
- Unit prices range from **-£53,594.36** to **£38,970.00**.
- The median transaction quantity is **3 units**, while the average is **9.94 units**, indicating a right-skewed distribution.
- The median product price is **£2.10**, with an average price of **£4.65**, suggesting that a relatively small number of high-value transactions influence the mean.

The presence of both negative quantities and negative prices suggests the existence of product returns, cancellations, refunds, or accounting adjustments that require further investigation before analysis.

---

### Categorical Data Assessment

The categorical variables indicate that:

- The dataset contains **53,628 unique invoices**.
- There are **5,305 unique product codes**.
- Product descriptions slightly exceed the number of product codes, suggesting that some products may have multiple descriptions or naming inconsistencies.
- Customers originate from **43 countries**, with the **United Kingdom accounting for approximately 92% of all transactions**, indicating that the dataset is heavily concentrated in the UK market.

---

### Initial Data Quality Findings

The profiling exercise identified several issues requiring attention during data cleaning:

- Invoice dates must be converted to datetime format.
- Customer records with missing customer identifiers require appropriate handling.
- Cancelled transactions (identified by invoice numbers beginning with **"C"**) should be investigated.
- Negative quantities and negative prices require validation and appropriate treatment.
- Duplicate records should be identified and removed where necessary.
- Feature engineering will be required to create variables supporting cohort analysis and CLTV calculations.

---

### Conclusion

The data profiling exercise confirms that the dataset is comprehensive and appropriate for customer retention and lifetime value analysis. While the transactional data is largely complete, several data quality issues—including missing customer identifiers, cancelled transactions, negative values, and date formatting—must be addressed before cohort analysis can be performed.

The findings from this profiling stage will directly inform the data cleaning, preprocessing, and feature engineering activities undertaken in the subsequent phase of the project.

## Data Quality Assessment Summary

### Overview

A comprehensive data quality assessment was conducted to identify issues that could affect the accuracy and reliability of the cohort retention and Customer Lifetime Value (CLTV) analyses. The assessment focused on missing values, duplicate records, cancelled transactions, invalid numerical values, and potential data inconsistencies.

---

## Missing Values

| Column | Missing Values | Percentage |
|---------|---------------:|-----------:|
| Customer ID | 243,007 | 22.77% |
| Description | 4,382 | 0.41% |

### Key Findings

- **Customer ID** contains a substantial number of missing values (22.77%). Since cohort analysis and CLTV calculations require tracking individual customers over time, transactions without customer identifiers cannot be assigned to a cohort and will be excluded from customer-level analyses.
- **Description** contains only 0.41% missing values. As product descriptions are not essential for retention or CLTV calculations, these records can be retained provided the remaining fields are valid.

---

## Duplicate Records

| Metric | Value |
|---------|------:|
| Duplicate Rows | 34,335 |

### Key Findings

Approximately **3.22%** of the dataset consists of duplicate records. These duplicates could lead to inflated transaction counts, revenue overestimation, and biased retention metrics if not removed. Exact duplicate records will therefore be eliminated during preprocessing.

---

## Cancelled Transactions

| Metric | Value |
|---------|------:|
| Cancelled Transactions | 19,494 |
| Percentage | 1.83% |

### Key Findings

Cancelled transactions are identified by invoice numbers beginning with the letter **"C"**. These transactions represent order cancellations or returns and do not reflect completed purchases. They will be excluded from cohort and CLTV analyses to ensure that only successful customer purchases contribute to retention and revenue calculations.

---

## Quantity Assessment

| Metric | Value |
|---------|------:|
| Negative Quantities | 22,950 |

### Key Findings

Negative quantities generally indicate returned or cancelled items. Their close correspondence with the number of cancelled invoices suggests that these records represent reverse transactions rather than valid purchases. These observations support excluding cancelled transactions from subsequent analyses.

---

## Price Assessment

| Metric | Value |
|---------|------:|
| Negative Prices | 5 |
| Zero Price Transactions | 6,202 |

### Key Findings

Only **five transactions** contain negative prices, indicating rare anomalies or accounting adjustments that require removal or further investigation.

Additionally, **6,202 transactions** have a unit price of zero. These may represent promotional items, free samples, replacement products, or data entry issues. These records will be examined before determining whether they should be retained or excluded from revenue-based analyses.

---

## Overall Data Quality Assessment

Overall, the dataset demonstrates good data quality and is well suited for customer retention and CLTV analysis. Most variables are complete, and the identified quality issues are typical of transactional retail datasets.

The principal data quality challenges include:

- Missing customer identifiers
- Duplicate transaction records
- Cancelled invoices
- Negative transaction quantities
- A small number of negative prices
- Zero-priced transactions
- Invoice dates requiring conversion to datetime format

---

## Data Cleaning Strategy

The following preprocessing actions will be implemented prior to cohort analysis:

| Data Quality Issue | Planned Action | Business Justification |
|--------------------|----------------|------------------------|
| Missing Customer ID | Remove | Customer cannot be assigned to a cohort or tracked over time. |
| Duplicate Records | Remove | Prevent double-counting of transactions and revenue. |
| Cancelled Invoices | Remove | Do not represent completed customer purchases. |
| Negative Quantities | Remove with cancelled transactions | Represent returned or reversed transactions. |
| Negative Prices | Remove | Invalid for revenue and CLTV calculations. |
| Zero Price Transactions | Investigate before deciding | May represent promotions or non-revenue transactions. |
| InvoiceDate | Convert to datetime | Required for cohort assignment and time-based analysis. |

---

## Conclusion

The data quality assessment indicates that the dataset is suitable for cohort retention and Customer Lifetime Value analysis following a structured preprocessing workflow. Implementing the planned cleaning procedures will improve data consistency, eliminate invalid observations, and ensure that subsequent analyses accurately reflect genuine customer purchasing behaviour.

## Data Cleaning & Feature Engineering Workflow

The data cleaning and feature engineering process follows a structured workflow to ensure data quality, consistency, and readiness for cohort retention and Customer Lifetime Value (CLTV) analysis.

```text
                    Raw Dataset
                         │
                         ▼
            1. Create Working Copy
                         │
                         ▼
           2. Convert Data Types
                         │
                         ▼
           3. Remove Duplicate Records
                         │
                         ▼
      4. Handle Missing Values
                         │
                         ▼
     5. Remove Invalid Transactions
                         │
                         ▼
          6. Feature Engineering
                         │
                         ▼
          7. Final Data Validation
                         │
                         ▼
         8. Save Clean Dataset
```

### Workflow Description

| Step | Activity | Purpose |
|------|----------|---------|
| **1** | Create Working Copy | Preserve the original dataset and perform all transformations on a separate copy. |
| **2** | Convert Data Types | Convert variables such as `InvoiceDate` to the appropriate data type for time-based analysis. |
| **3** | Remove Duplicate Records | Eliminate duplicate observations to prevent double-counting of transactions. |
| **4** | Handle Missing Values | Address missing customer identifiers and other incomplete records based on business requirements. |
| **5** | Remove Invalid Transactions | Exclude cancelled invoices, invalid quantities, and incorrect pricing to ensure analytical accuracy. |
| **6** | Feature Engineering | Create derived variables such as `TotalSales`, `InvoiceMonth`, `CohortMonth`, and other fields required for cohort and CLTV analysis. |
| **7** | Final Data Validation | Verify that the cleaned dataset satisfies all data quality requirements before analysis. |
| **8** | Save Clean Dataset | Export the cleaned and transformed dataset for use in subsequent cohort and CLTV analysis notebooks. |

The completion of this workflow produces a reliable analytical dataset that serves as the foundation for customer retention analysis, cohort matrix generation, and Customer Lifetime Value (CLTV) modelling.

In [19]:
# ==========================================
# Create Working Copy
# ==========================================

df_clean = df.copy()

print(f"Original Shape : {df.shape}")
print(f"Working Shape  : {df_clean.shape}")

Original Shape : (1067371, 8)
Working Shape  : (1067371, 8)


In [20]:
# ==========================================
# Convert InvoiceDate to DateTime
# ==========================================

df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

print(df_clean["InvoiceDate"].dtype)

datetime64[us]


In [21]:
# ==========================================
# Remove Duplicate Records
# ==========================================

initial_rows = len(df_clean)

df_clean.drop_duplicates(inplace=True)

removed_duplicates = initial_rows - len(df_clean)

print(f"Duplicate rows removed : {removed_duplicates:,}")
print(f"Remaining rows         : {len(df_clean):,}")

Duplicate rows removed : 34,335
Remaining rows         : 1,033,036


In [22]:
# ==========================================
# Remove Missing Customer IDs
# ==========================================

initial_rows = len(df_clean)

df_clean = df_clean.dropna(subset=["Customer ID"])

removed_missing_customer = initial_rows - len(df_clean)

print(f"Rows removed : {removed_missing_customer:,}")
print(f"Remaining    : {len(df_clean):,}")

Rows removed : 235,151
Remaining    : 797,885


In [23]:
# ==========================================
# Remove Cancelled Invoices
# ==========================================

initial_rows = len(df_clean)

df_clean = df_clean[
    ~df_clean["Invoice"].str.startswith("C")
]

removed_cancelled = initial_rows - len(df_clean)

print(f"Cancelled invoices removed : {removed_cancelled:,}")
print(f"Remaining rows             : {len(df_clean):,}")

Cancelled invoices removed : 18,390
Remaining rows             : 779,495


In [24]:
# ==========================================
# Remove Invalid Prices
# ==========================================

initial_rows = len(df_clean)

df_clean = df_clean[df_clean["Price"] > 0]

removed_prices = initial_rows - len(df_clean)

print(f"Rows removed : {removed_prices:,}")
print(f"Remaining    : {len(df_clean):,}")

Rows removed : 70
Remaining    : 779,425


In [25]:
# ==========================================
# Remove Negative Quantities
# ==========================================

initial_rows = len(df_clean)

df_clean = df_clean[df_clean["Quantity"] > 0]

removed_qty = initial_rows - len(df_clean)

print(f"Rows removed : {removed_qty:,}")
print(f"Remaining    : {len(df_clean):,}")

Rows removed : 0
Remaining    : 779,425


In [26]:
# ==========================================
# Final Data Quality Check
# ==========================================

print("=" * 60)
print("FINAL DATA QUALITY CHECK")
print("=" * 60)

print(f"Missing Customer IDs : {df_clean['Customer ID'].isna().sum():,}")
print(f"Duplicate Rows       : {df_clean.duplicated().sum():,}")
print(f"Negative Quantity    : {(df_clean['Quantity'] < 0).sum():,}")
print(f"Negative Price       : {(df_clean['Price'] < 0).sum():,}")
print(f"Zero Price           : {(df_clean['Price'] == 0).sum():,}")
print(f"Cancelled Invoices   : {df_clean['Invoice'].str.startswith('C').sum():,}")

print("\nFinal Dataset Shape")
print(df_clean.shape)

FINAL DATA QUALITY CHECK
Missing Customer IDs : 0
Duplicate Rows       : 0
Negative Quantity    : 0
Negative Price       : 0
Zero Price           : 0
Cancelled Invoices   : 0

Final Dataset Shape
(779425, 8)


In [27]:
print(df_clean.shape)

print("Negative Prices:", (df_clean["Price"] < 0).sum())
print("Zero Prices:", (df_clean["Price"] == 0).sum())
print("Negative Quantity:", (df_clean["Quantity"] < 0).sum())
print("Cancelled:", df_clean["Invoice"].str.startswith("C").sum())
print("Duplicates:", df_clean.duplicated().sum())
print("Missing Customer:", df_clean["Customer ID"].isna().sum())

(779425, 8)
Negative Prices: 0
Zero Prices: 0
Negative Quantity: 0
Cancelled: 0
Duplicates: 0
Missing Customer: 0


In [28]:
print(df_clean.shape)

print(df_clean.head())

print(df_clean.tail())

(779425, 8)
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  
0 2009-12-01 07:45:00   6.95    13,085.00  United Kingdom  
1 2009-12-01 07:45:00   6.75    13,085.00  United Kingdom  
2 2009-12-01 07:45:00   6.75    13,085.00  United Kingdom  
3 2009-12-01 07:45:00   2.10    13,085.00  United Kingdom  
4 2009-12-01 07:45:00   1.25    13,085.00  United Kingdom  
        Invoice StockCode                      Description  Quantity  \
1067366  581587     22899     CHILDREN'S APRON DOLLY GIRL          6   
1067367  581587     23254    CHILDRENS CUTLERY DOLLY GIRL      

In [29]:
print(df_clean["Invoice"].str.startswith("C").sum())
print((df_clean["Quantity"] <= 0).sum())
print((df_clean["Price"] <= 0).sum())
print(df_clean["Customer ID"].isna().sum())
print(df_clean.duplicated().sum())

0
0
0
0
0


In [30]:
cleaning_log = []

# Remove duplicates
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)

cleaning_log.append({
    "Step": "Remove Duplicates",
    "Rows Removed": before - after,
    "Rows Remaining": after
})

In [31]:
pd.DataFrame(cleaning_log)

,Step,Rows Removed,Rows Remaining
0,Remove Duplicates,0,779425


## Data Cleaning Summary

The raw transactional dataset underwent a structured data cleaning process to improve data quality and prepare it for cohort retention and Customer Lifetime Value (CLTV) analysis.

The cleaning process included:

- Creating a working copy of the original dataset.
- Converting `InvoiceDate` to a datetime data type.
- Removing duplicate transaction records.
- Excluding transactions with missing customer identifiers.
- Removing cancelled invoices.
- Eliminating transactions with non-positive quantities and prices.
- Performing final validation to ensure all identified data quality issues had been resolved.

### Dataset Size Before and After Cleaning

| Stage | Rows | Columns |
|-------|-----:|--------:|
| Raw Dataset | 1,067,371 | 8 |
| Cleaned Dataset | 779,425 | 8 |

### Final Data Quality Validation

| Validation Check | Result |
|------------------|-------:|
| Missing Customer IDs | 0 |
| Duplicate Records | 0 |
| Cancelled Transactions | 0 |
| Negative Quantities | 0 |
| Negative Prices | 0 |
| Zero Price Transactions | 0 |

The resulting dataset is complete, internally consistent, and suitable for customer retention, cohort analysis, and Customer Lifetime Value (CLTV) modelling.

In [32]:
# ==========================================
# Save Clean Dataset
# ==========================================

output_path = PROCESSED_DATA / "online_retail_clean.csv"

df.to_csv(output_path, index=False)

print("✅ Clean dataset saved successfully.")
print(f"Location: {output_path}")

✅ Clean dataset saved successfully.
Location: c:\Users\HP\OneDrive\Documents\My Git Project\SaaS-ECommerce-Cohort-Retention-CLTV-Analysis\data\processed\online_retail_clean.csv


In [33]:
output_path.exists()

True

In [34]:
saved_df = pd.read_csv(output_path)

print(saved_df.shape)

saved_df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom
